# Implementation of a transformer model
In this exercise we will implement the transformer model (an architecture based on LLama2). The squeleton of the code is provided and you should complete it (mostly self attention mechanism).

In [3]:
import time 
from time import gmtime, strftime
import math
import shutil
import os



# neural network utilities
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.utils.data import DataLoader

# dataset an tokenization
from datasets import load_dataset
from transformers import AutoTokenizer

from rope_embedding import RoPE


## Self Attention with no mask

Let consider $Q, K, V \in R^{B\times H \times L \times N}$ for each batch element (B) for each heads (H) we want to compute $$A = \frac{Q_{bh}.K_{bh}^{t}}{\sqrt{N}}$$

Attention weights are defined by $$ 
softmax(A_i) = \begin{pmatrix}
\frac{e^{A_{i1}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} & 
\frac{e^{A_{i2}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} & 
\dots &
\frac{e^{A_{iL}}}{\sum\limits_{j=1}^L  e^{A_{ij}}}
\end{pmatrix}$$ 
And $$ 
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} & 
\frac{e^{A_{12}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} & 
\dots &
\frac{e^{A_{1L}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} & 
\frac{e^{A_{22}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} & 
\dots &
\frac{e^{A_{2l}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\dots &
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}}
\end{pmatrix}$$ 


Notice that the function softmax can be used to compute this matrix [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html#torch.softmax).


## Masked Attention
In many cases, certains tokens will not compute attention with later tokens (padding or decoding approaches). Typically in decoder only architectures we would have the attention weights matrix only considering lower diagonal values :

$$ 
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^1  e^{A_{1j}}} & 
0 & 
\dots & 0 &
0 \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} & 
\frac{e^{A_{22}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} & 
\dots & 0 &
0 \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{{L-1}1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} & 
\frac{e^{A_{{L-1}2}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} & 
\dots &
\frac{e^{A_{L-1L-1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}}& 0 \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\dots &
\frac{e^{A_{LL-1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} \\
\end{pmatrix}$$ 

A simple implementation would consist in setting the upper diagonal of A to $-\infty$ and apply the softmax. We will thus consider a matrix name $M$ (or `mask` in the code) having upper diagonal element to $-\infty$ : 
$$ M = 
\begin{pmatrix}
0& 
-\infty & 
\dots & -\infty&
-\infty \\
0 & 
0 & 
\dots & -\infty &
-\infty \\
\vdots & \vdots & \ddots & \vdots \\
0 & 
0 & 
\dots &
0& -\infty \\
0 & 
0 & 
\dots &
0 & 
0\\
\end{pmatrix}
$$

And apply softmax on $A + M$

In [55]:
def multi_head_attention(
    q : torch.FloatTensor,
    k : torch.FloatTensor,
    v : torch.FloatTensor,
    mask : torch.FloatTensor
):
    """
    Given Q,K,V and the attention mask compute cross attention

    The shape of the matrix are the following:
    B: The batch size (number of examples)
    Q: The size of the decoder sequence
    K: The size of the encoder sequence
    H: Then number of attention heads
    N: The embedding size 
    
    Parameters:
        q : torch.FloatTensor
            The query matrix of shape BxHxQxN
        k : torch.FloatTensor
            The key matrix of shape BxHxKxN
        v : torch.FloatTensor
            The value matrix of shape BxHxKxN
        mask : torch.FloatTensor
            The attention mask of shape BxQxK,
            the masked elements are set to -inf 
            else elements are set to 0
    Return : (torch.Tensor, torch.Tensor)
        return two tensor the first containing 
        the attention weights (QK^t) and the second 
        the result of the attention
    """
    B = q.shape[0]
    H = q.shape[1]
    Q = q.shape[2]
    N = q.shape[3]
    K = k.shape[2]

    batch_weights, batch_output = [], []
    for b in range(B): 
        
        head_weights, head_output = [], []
        for h in range(H): 

            Q_bh =  q[b,h] # dim (Q, N)
            K_bh = k[b, h] # dim (K, N)
            V_bh = v[b, h] # dim (K, N)
            m = mask[b] # dim (Q, K)
            
            #print(Q_bh.size())
            #print(K_bh.size())
            #print(V_bh.size())
            #print(m.size())

            K_bh_t = torch.transpose(K_bh, 0, 1)
            A = torch.matmul(Q_bh, K_bh_t)
            A_norm = (A + m)  / math.sqrt(N) # adding of the mask 

            A_soft = torch.softmax(A_norm, dim=-1) # -1 for the last dimension 

            output = torch.matmul(A_soft, V_bh)

            head_weights.append(A_soft)
            head_output.append(output)

        batch_weights.append(torch.stack(head_weights))
        batch_output.append(torch.stack(head_output))

    attention_weights = torch.stack(batch_weights)
    attention_outputs = torch.stack(batch_output)
    
    return attention_weights, attention_outputs

In [54]:
# test to run 
q, k, v = (torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4))
mask = torch.triu(-q.new_ones(1, 3, 3) *  torch.inf, diagonal =  1)
attention_weigths, attention_output = multi_head_attention(q, k, v, mask)


assert(torch.all(attention_weigths == torch.Tensor([
          [[[1., 0., 0.],
            [1/2, 1/2, 0.],
            [1/3, 1/3, 1/3]]]])))


torch.Size([3, 4])
torch.Size([3, 4])
torch.Size([3, 4])
torch.Size([3, 3])


##  Create the attention module

In pytorch module are block of neural networks that inherit from the class `nn.Module`.  In the following you willbe asked to complete the attention code. The attention module compute the Query $Q$, the Keys $K$ and Values $V$ for each heads and compute attention. The forward methods is the method implementing these operations. In this lab we use a mult-head attention, meaning that we compute attention for each heads, that can be done with a for loop or using batched operation (it is up to you). Also as we follow the Llama 2 architecture, we use for positional embeding the RoPe positonal encoding that is applied on the two matrix $Q$ and $K$. The forward function should compute for an input $X \in \mathbb{R}^{L \times N}$ (in the decoder only case) the following:

1. $Q_h = RoPE(W^q_{h}X^{\intercal})$ (for all heads h) with $W^q_h \in \mathbb{R}^{N \times N}$
2. $K_h = RoPE(W^k_{h}X^{\intercal})$ (for all heads h)
3. $V_h = W^v_{h}X^{\intercal}$ (for all heads h)
4. $A_h =  Attention(Q_h, K_h, V_h)$ (for all heads h)
5. $A$ = [A_1, A_2, \dots A_H] (the concatenation along the last dimension)
6. $O = W^oA^{\intercal}$ with $w^{o} \in \mathbb{R}^{N \times NH}$ (NH being the scalar multplication of N and H)

Notice that Q, K and V can be computed without a loop using $W_{q}$ (respectively for $W_k$ and $W_v$) .


In [ ]:

class RoPEAttentionModule(nn.Module):
    """ A cross/self attention pytorch module.
    
    """
    def __init__(self, input_dim, output_dim, num_heads=1):
        super().__init__()
        
        self.Wq, self.Wk, self.Wv  =\
            (nn.Linear(input_dim, output_dim * num_heads) for _ in range(3))

        self.output_dim = output_dim
        self.num_heads = num_heads
        self.Wo = nn.Linear(output_dim * num_heads, input_dim)
        self.rope_func = RoPE(output_dim)
    
    def forward(
            self,
            x : torch.Tensor,
            attention_mask : torch.BoolTensor = None, 
            y : torch.Tensor = None,
            decoder_mask = True,
            k_cache = None,
            v_cache = None,
            start_cache = 0
        ):
        ''' 
            Parameters:
                x : torch.Tensor
                    The input of the Attention module
                    used at least for K, V computation
                    (for Q if decoder only)
                attention_mask : torch.BoolTensor
                    The mask for attention
                y : torch.Tensor or None
                    The query input in the case of
                    cross-attention
        '''
        for h in range(self.num_heads): 
            Q_h = self.Wq[h]*
        return attention_weigths, output

    


    



## Trasnsformer FeedForward

The FeedForward Network is defined using Llama 2 architecture as following for $X \in \mathbb{R^{L\times N}}$ :


1. $G = SiLU(W^gX^\intercal)$ with $W^g \in \mathbb{R}^{M \times N}$ (see SiLU in pytorch [documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html))
2. $U = W^u X^\intercal$ with $W^u  \in \mathbb{R}^{M \times N}$
3. $I = G \odot I$ (Hadamard multiplication)
4. $O = W^{o} I^{\intercal}$ with $W_{o} \in \mathbb{R}^{N \times M}$






In [ ]:
class TransformerFeedForward(nn.Module):
    def __init__(self, embed_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(embed_size, intermediate_size)
        self.up_proj = nn.Linear(embed_size, intermediate_size)
        self.down_proj = nn.Linear(intermediate_size, embed_size)
        self.gate_func = nn.SiLU()
        
    def forward(self, x):
        raise NotImplementedError

## Transformer block
We can now create the block of the 

Let consider $x_0 \in \mathbb{R}^{B \times L \times N}$, the decoder block is given by : 

* $x_1 = LN_1(x_0)$ (apply layer norm)
* $x_2 = Attention(x_1)$ (apply attention)
* $x_3 = x_0 + x_2$ (adding residual)
* $x_4 = LN_2(x_3)$
* $y = FF(x_4) + x_4$


In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_size, intermediate_size, num_heads):
        super().__init__()
        self.attention_module =\
            RoPEAttentionModule(embed_size, embed_size, num_heads=num_heads)
        self.feed_forward = TransformerFeedForward(embed_size, intermediate_size)
        
        self.attention_layer_norm = nn.RMSNorm(embed_size)
        self.feed_forward_layer_norm = nn.RMSNorm(embed_size)
        
    def forward(
        self,
        x,
        attention_mask = None,
        k_cache=None,
        v_cache=None,
        start_cache=0,
        layer=0,
    ):
        
        raise NotImplementedError
        
        return output

## The Model

The model contain a stack of block on input embedding, it applies the different transformation and should return both the output embeddings and the logits (size of the vocabulary).

It should return $o_{e} \mathbb{R}^{B\times L \times N}$ and $\hat{y}\in \mathbb{R}^{B\times L \times V}$ with : 

* $o_e = RMSNORM \left( b_h\circ \dots   \circ \circ b_2 \circ b_1 (w_{word\_embedding}x^\intercal) \right)$ with $b_i$ the $i^{th}$ block
* $\hat{y} = W_{lm\_head} o_e^{T}$



In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocabulary_size, embed_size, intermediate_size, num_heads, hidden_layers=5, **kwargs):
        super().__init__(**kwargs)
        self.nh = num_heads
        self.nhl = hidden_layers
        self.es = embed_size 

        self.wte = nn.Embedding(vocabulary_size, embed_size)

        self.blocks = nn.ModuleList([TransformerDecoderBlock(embed_size, intermediate_size, num_heads) for _ in range(hidden_layers)])
        self.ln_f = nn.LayerNorm(embed_size, eps=1e-3)
        self.lm_head = nn.Linear(embed_size, vocabulary_size)


    def forward(
            self,
            input_ids,
            attention_mask = None,
            k_cache = None,
            v_cache = None,
            start_cache = 0
        ):
        

        input_embed = self.wte(input_ids) 
        intermediate = input_embed 

        for i, block in enumerate(self.blocks):
            raise NotImplementedError

        
        output_embed = self.ln_f(intermediate)
        output_lm = self.lm_head(output_embed)  

        return output_embed, output_lm

##  Training the transformer

### I.Dataset 
For simplicity we will use a small dataset name TinyStories, we can load it using the huggingface Datasets library as following

In [ ]:

tinystories_dataset = load_dataset("roneneldan/TinyStories")
training_set = tinystories_dataset['train']
print(training_set[0])

### The Dataloader

We can define the batch_size default we will use, using the DataLoader object. Here we consider that the training machine will have enough RAM for a batch of 16 samples.

In [ ]:
training_dl = DataLoader(training_set, batch_size=16, shuffle=True) 

### III. Tokenizer

In this exercice we will consider the LLama Tokenizer, however, you can train your own tokenizer if you prefer (see lab 2)

In [ ]:


os.environ["TOKENIZERS_PARALLELISM"] = "false"

model_hg_id = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_hg_id)
tokenizer.pad_token = '<pad>'
vocabulary_size = tokenizer.vocab_size

### IV. Create the model and the optimizer

In [ ]:
device = 'cuda'
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 2)
model = model.train()
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=6e-4)


In [ ]:
model

### VI. Training the model

In [ ]:
model = model.train()
avg_loss = []
start_time = time.time()

for i, data in enumerate(training_dl):
    res = tokenizer(data['text'], return_tensors="pt", padding=True, padding_side='right', max_length=128, truncation=True)
    x = res.input_ids[:, :-1] + 0
    y = res.input_ids[:, 1:] + 0
    # attention_mask = (res.attention_mask == 0)
    optimizer.zero_grad()
    oe, oy = model(x.to(device))
    loss = loss_function(oy.to(device).view(-1, vocabulary_size), y.to(device).view(-1))
    loss.backward()
    loss_value = loss.item()
    avg_loss.append(loss_value)
    if( i%500 == 0):
        elapsed_time = time.time() - start_time
        remaining_time = int((elapsed_time/(i+1)) * (len(training_dl) - i))
        loop_sec = ((i+1)/elapsed_time)
        print(f"The loss at iteration {i+1} is {sum(avg_loss)/len(avg_loss):3.4f} remaining_time is {strftime('%H:%M:%S', gmtime(remaining_time))}s ({loop_sec:4.1f} it/s)", flush=True)
        avg_loss = []
    if( i%1000 == 0):
        torch.save(optimizer.state_dict() ,"optimizer_state_dict_llama_mini.pth.temp")
        torch.save(model.state_dict() ,"transformer_state_dict_llama_mini.pth.temp")
        shutil.copyfile("optimizer_state_dict_llama_mini.pth.temp", "optimizer_state_dict_llama_mini.pth")
        shutil.copyfile("transformer_state_dict_llama_mini.pth.temp", "transformer_state_dict_llama_mini.pth")
    optimizer.step()

## Decoding time

There is different way to decode, the simplest one is the greedy decoding method, the principle is to loop with the new token produced each time (complete `nonefficient_greedy_decoding`) and choosing the token the most likely. We can also store Keys and Values as shown in course that in practice only accellerate the generation but not change it (complete `greedy_decoding`). And finally we can also sampling new tokens (complete `sampling_decoding` ).

In [ ]:
def nonefficient_greedy_decoding(model, x, max_new_tokens=64):
    previous_gen = x
    for _ in range(max_new_tokens):
        raise NotImplementedError
    
    return previous_gen

def greedy_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=None):
    raise NotImplementedError

def sampling_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=None, temperature=.7):

    raise NotImplementedError


In [ ]:
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 2)
model.load_state_dict(torch.load("transformer_state_dict_llama_mini.pth", map_location="cpu"))
model = model.eval()

decoding_method = greedy_decoding

text = "Alice"
model.to("cpu")
tokenized_text = tokenizer(text, return_tensors='pt')

output_ids = decoding_method(model, tokenized_text.input_ids.to("cpu"), max_new_tokens=64)[0]
print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0])

## Evaluation ?

It remains difficult to evaluate the model. You can here propose an evalaution based on perplexity using an other model, or comparing results to ground truth using the test (here only validation) set, but it will not be totally informative !!!

In [ ]:
validation_dataset = tinystories_dataset['validation']